In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from tqdm import tqdm
import warnings
import time
from datetime import datetime
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# Machine Learning
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    roc_auc_score
)
import xgboost as xgb

# Set seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# GPU Configuration
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU available: {len(gpus)} device(s)")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected, using CPU")


class ImageDataGenerator:
    """Memory-efficient image data generator for fine-tuned ResNet"""

    def __init__(self, sample_info, indices, image_size, batch_size=32):
        self.sample_info = sample_info
        self.indices = indices
        self.image_size = image_size
        self.batch_size = batch_size

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        images = []
        for i in batch_indices:
            img = load_img(self.sample_info[i]['image_path'], target_size=self.image_size)
            img_array = img_to_array(img)
            img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
            images.append(img_array)

        return np.array(images)

    def get_full_dataset(self):
        """Load entire dataset (use only for small datasets)"""
        all_images = []
        for i in tqdm(self.indices, desc="Loading images"):
            img = load_img(self.sample_info[i]['image_path'], target_size=self.image_size)
            img_array = img_to_array(img)
            img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
            all_images.append(img_array)
        return np.array(all_images)


class MultimodalBrainTumorClassifier:
    """Multimodal classifier with computational efficiency tracking"""

    def __init__(self, csv_path, data_root, image_size=(224, 224)):
        """
        Args:
            csv_path: Path to CSV with radiomics features
            data_root: Root directory containing glioma/, meningioma/, pituitary/
            image_size: Input size for ResNet50
        """
        self.csv_path = csv_path
        self.data_root = Path(data_root)
        self.image_size = image_size

        # Data containers
        self.df = None
        self.radiomics_features = None
        self.deep_features = None
        self.deep_features_with_mask = None  # NEW: Store mask-guided features
        self.labels = None
        self.sample_info = []

        # Encoders
        self.label_encoder = LabelEncoder()

        # Models
        self.resnet_feature_extractor = None
        self.fine_tuned_model = None

        self.results = {}
        self.cv_predictions = {}
        self.roc_data = {}

        # Computational efficiency tracking
        self.computation_stats = {}

    def scan_image_directory(self):
        """Scan directory structure and create mapping"""
        print("=" * 80)
        print("SCANNING IMAGE DIRECTORY")
        print("=" * 80)

        tumor_types = ['glioma', 'meningioma', 'pituitary']
        image_mapping = {}

        for tumor_type in tumor_types:
            tumor_dir = self.data_root / tumor_type

            if not tumor_dir.exists():
                print(f"⚠ Warning: {tumor_type} directory not found at {tumor_dir}")
                continue

            image_files = list(tumor_dir.glob("*_image.png"))
            print(f"\n{tumor_type:12s}: {len(image_files)} images found")

            for image_path in image_files:
                sample_id = image_path.stem.replace('_image', '')
                mask_path = image_path.parent / f"{sample_id}_mask.png"

                if not mask_path.exists():
                    mask_path = None

                image_mapping[sample_id] = {
                    'tumor_type': tumor_type,
                    'image_path': image_path,
                    'mask_path': mask_path
                }

        print(f"\n✓ Total images mapped: {len(image_mapping)}")
        return image_mapping

    def load_and_prepare_data(self):
        """Load CSV and match with images"""
        print("\n" + "=" * 80)
        print("LOADING AND MATCHING DATA")
        print("=" * 80)

        self.df = pd.read_csv(self.csv_path)
        print(f"✓ Loaded CSV: {self.df.shape}")

        duplicates = self.df['sample_id'].duplicated()
        if duplicates.any():
            dup_ids = self.df[duplicates]['sample_id'].unique()
            print(f"\n⚠ WARNING: Found {len(dup_ids)} duplicate sample_id(s): {dup_ids}")
            print("   Keeping first occurrence of each duplicate...")
            self.df = self.df.drop_duplicates(subset='sample_id', keep='first')
            print(f"   Deduplicated CSV: {self.df.shape}")

        image_mapping = self.scan_image_directory()

        exclude_cols = ['sample_id', 'class_name', 'label']
        feature_cols = [col for col in self.df.columns if col not in exclude_cols]

        matched_samples = []
        matched_features = []
        matched_labels = []

        print("\nMatching CSV with images...")
        for idx, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Matching"):
            sample_id = str(row['sample_id'])

            if sample_id in image_mapping:
                matched_samples.append({
                    'sample_id': sample_id,
                    'csv_class': row['class_name'],
                    'dir_class': image_mapping[sample_id]['tumor_type'],
                    'image_path': image_mapping[sample_id]['image_path'],
                    'mask_path': image_mapping[sample_id]['mask_path']
                })
                matched_features.append(row[feature_cols].values)
                matched_labels.append(row['class_name'])

        print(f"\n✓ Matched samples: {len(matched_samples)}/{len(self.df)}")

        if len(matched_samples) == 0:
            print("\n ERROR: No samples matched between CSV and images!")
            return None

        self.sample_info = matched_samples
        self.radiomics_features = np.array(matched_features)
        self.labels = self.label_encoder.fit_transform(matched_labels)

        print(f"\n✓ Final dataset:")
        print(f"   Samples: {len(self.labels)}")
        print(f"   Radiomics Features: {self.radiomics_features.shape[1]}")
        print(f"   Classes: {self.label_encoder.classes_}")
        print(f"\n✓ Class distribution:")
        for cls, count in zip(*np.unique(self.labels, return_counts=True)):
            print(f"   {self.label_encoder.classes_[cls]:12s}: {count}")

        return self

    def extract_deep_features(self, batch_size=32, use_mask=False):
        """
        Extract deep features from ResNet50 (frozen)

        Args:
            batch_size: Number of images to process at once
            use_mask: Whether to apply tumor masks
        """
        print("\n" + "=" * 80)
        print(f"EXTRACTING DEEP FEATURES (ResNet50 - Frozen, use_mask={use_mask})")
        print("=" * 80)

        start_time = time.time()

        base_model = ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.image_size, 3),
            pooling='avg'
        )
        self.resnet_feature_extractor = base_model

        print(f"ResNet50 loaded (output shape: {base_model.output_shape})")
        print(f"Using batch size: {batch_size}")

        if use_mask:
            print("Will apply masks to focus on tumor regions")

        features_list = []
        n_samples = len(self.sample_info)
        n_batches = int(np.ceil(n_samples / batch_size))

        print("\nExtracting features from images...")
        for batch_idx in tqdm(range(n_batches), desc="Processing batches"):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, n_samples)
            batch_samples = self.sample_info[start_idx:end_idx]

            batch_images = []
            for sample in batch_samples:
                try:
                    img = load_img(sample['image_path'], target_size=self.image_size)
                    img_array = img_to_array(img)

                    if use_mask and sample['mask_path'] is not None:
                        mask = cv2.imread(str(sample['mask_path']), cv2.IMREAD_GRAYSCALE)
                        mask = cv2.resize(mask, self.image_size)
                        mask = mask / 255.0
                        mask = np.expand_dims(mask, axis=-1)
                        img_array = img_array * mask

                    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
                    batch_images.append(img_array)

                except Exception as e:
                    print(f"⚠ Error processing {sample['sample_id']}: {str(e)}")
                    batch_images.append(np.zeros((*self.image_size, 3)))

            # Process batch
            batch_array = np.array(batch_images)
            batch_features = self.resnet_feature_extractor.predict(batch_array, verbose=0)

            for features in batch_features:
                features_list.append(features.flatten())

        extracted_features = np.array(features_list)

        # Store features based on whether mask was used
        if use_mask:
            self.deep_features_with_mask = extracted_features
            print(f"\n✓ Deep features (with mask) extracted: {self.deep_features_with_mask.shape}")
        else:
            self.deep_features = extracted_features
            print(f"\n✓ Deep features (without mask) extracted: {self.deep_features.shape}")

        extraction_time = time.time() - start_time

        print(f"✓ Extraction time: {extraction_time:.2f} seconds ({extraction_time/len(self.sample_info):.4f} sec/sample)")

        # Store computational stats
        mask_suffix = "_with_mask" if use_mask else "_without_mask"
        self.computation_stats[f'deep_feature_extraction{mask_suffix}'] = {
            'total_time': extraction_time,
            'time_per_sample': extraction_time / len(self.sample_info),
            'feature_dim': extracted_features.shape[1]
        }

        return self

    def create_xgboost_classifier(self):
        """Create XGBoost classifier (used for ALL approaches for fair comparison)"""
        return xgb.XGBClassifier(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric='mlogloss',
            verbosity=0
        )

    def create_fine_tuned_resnet(self, num_classes):
        """Create fine-tuned ResNet50 model"""
        print("\nCreating fine-tuned ResNet50 model...")

        base_model = ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.image_size, 3)
        )

        # Freeze first layers, unfreeze last layers for fine-tuning
        for layer in base_model.layers[:-30]:
            layer.trainable = False
        for layer in base_model.layers[-30:]:
            layer.trainable = True

        print(f"  Trainable layers: {sum([1 for l in base_model.layers if l.trainable])}/{len(base_model.layers)}")

        # Build model
        model = Sequential([
            base_model,
            GlobalAveragePooling2D(),
            BatchNormalization(),
            Dense(512, activation='relu'),
            Dropout(0.5),
            BatchNormalization(),
            Dense(256, activation='relu'),
            Dropout(0.3),
            Dense(num_classes, activation='softmax')
        ])

        model.compile(
            optimizer=Adam(learning_rate=1e-5),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        return model

    def calculate_metrics(self, y_true, y_pred, y_pred_proba, classes):
        """Calculate comprehensive metrics"""
        n_classes = len(classes)

        accuracy = accuracy_score(y_true, y_pred)

        precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
        recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
        f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)

        specificity_per_class = []
        for i in range(n_classes):
            tn = np.sum((y_true != i) & (y_pred != i))
            fp = np.sum((y_true != i) & (y_pred == i))
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            specificity_per_class.append(specificity)

        specificity_per_class = np.array(specificity_per_class)

        sensitivity_macro = np.mean(recall_per_class)
        specificity_macro = np.mean(specificity_per_class)
        precision_macro = np.mean(precision_per_class)
        f1_macro = np.mean(f1_per_class)

        try:
            y_true_bin = label_binarize(y_true, classes=range(n_classes))
            auc_per_class = []

            for i in range(n_classes):
                if len(np.unique(y_true_bin[:, i])) > 1:
                    try:
                        auc_score = roc_auc_score(y_true_bin[:, i], y_pred_proba[:, i])
                        auc_per_class.append(auc_score)
                    except Exception as e:
                        auc_per_class.append(np.nan)
                else:
                    auc_per_class.append(np.nan)

            auc_per_class = np.array(auc_per_class)
            auc_macro = np.nanmean(auc_per_class)

        except Exception as e:
            auc_per_class = np.full(n_classes, np.nan)
            auc_macro = np.nan

        return {
            'accuracy': accuracy,
            'sensitivity_macro': sensitivity_macro,
            'specificity_macro': specificity_macro,
            'precision_macro': precision_macro,
            'f1_macro': f1_macro,
            'auc_macro': auc_macro,
            'sensitivity_per_class': recall_per_class,
            'specificity_per_class': specificity_per_class,
            'auc_per_class': auc_per_class
        }

    def validate_cv_folds(self, y, n_splits):
        """Validate that all classes appear in each fold"""
        print(f"\nValidating {n_splits}-fold cross-validation")
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

        all_valid = True
        n_classes = len(np.unique(y))

        for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y)), y), 1):
            train_classes = len(np.unique(y[train_idx]))
            val_classes = len(np.unique(y[val_idx]))

            if train_classes != n_classes or val_classes != n_classes:
                print(f"Fold {fold}: Train has {train_classes}/{n_classes} classes, "
                      f"Val has {val_classes}/{n_classes} classes")
                all_valid = False
            else:
                print(f"   ✓ Fold {fold}: All {n_classes} classes present in both train and val")

        if not all_valid:
            print("Some folds are missing classes!")
        else:
            print("All folds valid - each contains all classes")

        return all_valid

    def evaluate_approach(self, approach_name, X, y, classifier_type='xgboost', n_splits=5):
        """Evaluate approach with CV and computational tracking"""
        print(f"\n{'=' * 80}")
        print(f"EVALUATING: {approach_name}")
        print(f"{'=' * 80}")

        self.validate_cv_folds(y, n_splits)

        approach_start_time = time.time()

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

        fold_metrics = []
        all_y_true = []
        all_y_pred = []
        all_y_pred_proba = []
        roc_curves_data = []

        training_times = []
        inference_times = []

        n_classes = len(np.unique(y))
        classes = self.label_encoder.classes_

        for fold, (train_idx, val_idx) in enumerate(skf.split(X if X is not None else np.zeros(len(y)), y), 1):
            print(f"\n--- Fold {fold}/{n_splits} ---")

            if classifier_type == 'xgboost':
                X_train, X_val = X[train_idx], X[val_idx]

            y_train, y_val = y[train_idx], y[val_idx]

            train_start = time.time()

            if classifier_type == 'xgboost':
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_val_scaled = scaler.transform(X_val)

                model = self.create_xgboost_classifier()
                model.fit(
                    X_train_scaled, y_train,
                    eval_set=[(X_val_scaled, y_val)],
                    verbose=False
                )

                train_time = time.time() - train_start

                inference_start = time.time()
                y_pred = model.predict(X_val_scaled)
                y_pred_proba = model.predict_proba(X_val_scaled)
                inference_time = time.time() - inference_start

            elif classifier_type == 'fine_tuned_resnet':
                print("   Loading images for this fold...")

                train_images = []
                val_images = []

                for idx in tqdm(train_idx, desc="   Train images", leave=False):
                    img = load_img(self.sample_info[idx]['image_path'], target_size=self.image_size)
                    img_array = img_to_array(img)
                    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
                    train_images.append(img_array)

                for idx in tqdm(val_idx, desc="   Val images", leave=False):
                    img = load_img(self.sample_info[idx]['image_path'], target_size=self.image_size)
                    img_array = img_to_array(img)
                    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
                    val_images.append(img_array)

                X_train_images = np.array(train_images)
                X_val_images = np.array(val_images)

                model = self.create_fine_tuned_resnet(n_classes)

                early_stop = EarlyStopping(
                    monitor='val_loss',
                    patience=10,
                    restore_best_weights=True,
                    verbose=0
                )

                reduce_lr = ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.5,
                    patience=5,
                    min_lr=1e-7,
                    verbose=0
                )

                model.fit(
                    X_train_images, y_train,
                    validation_data=(X_val_images, y_val),
                    epochs=50,
                    batch_size=32,
                    callbacks=[early_stop, reduce_lr],
                    verbose=0
                )

                train_time = time.time() - train_start

                inference_start = time.time()
                y_pred_proba = model.predict(X_val_images, verbose=0)
                y_pred = np.argmax(y_pred_proba, axis=1)
                inference_time = time.time() - inference_start

            training_times.append(train_time)
            inference_times.append(inference_time)

            metrics = self.calculate_metrics(y_val, y_pred, y_pred_proba, classes)
            fold_metrics.append(metrics)

            all_y_true.extend(y_val)
            all_y_pred.extend(y_pred)
            all_y_pred_proba.extend(y_pred_proba)

            y_val_bin = label_binarize(y_val, classes=range(n_classes))
            fold_roc = {}
            for i in range(n_classes):
                if len(np.unique(y_val_bin[:, i])) > 1:
                    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_pred_proba[:, i])
                    fold_roc[i] = {'fpr': fpr, 'tpr': tpr}
            roc_curves_data.append(fold_roc)

            print(f"  Accuracy: {metrics['accuracy']:.4f}")
            print(f"  AUC: {metrics['auc_macro']:.4f}")
            print(f"  Training time: {train_time:.2f}s")
            print(f"  Inference time: {inference_time:.4f}s ({inference_time/len(val_idx)*1000:.2f} ms/sample)")

        total_approach_time = time.time() - approach_start_time

        aggregated_metrics = {
            'accuracy_mean': np.mean([m['accuracy'] for m in fold_metrics]),
            'accuracy_std': np.std([m['accuracy'] for m in fold_metrics]),
            'sensitivity_mean': np.mean([m['sensitivity_macro'] for m in fold_metrics]),
            'sensitivity_std': np.std([m['sensitivity_macro'] for m in fold_metrics]),
            'specificity_mean': np.mean([m['specificity_macro'] for m in fold_metrics]),
            'specificity_std': np.std([m['specificity_macro'] for m in fold_metrics]),
            'f1_mean': np.mean([m['f1_macro'] for m in fold_metrics]),
            'f1_std': np.std([m['f1_macro'] for m in fold_metrics]),
            'auc_mean': np.nanmean([m['auc_macro'] for m in fold_metrics]),
            'auc_std': np.nanstd([m['auc_macro'] for m in fold_metrics]),
            'fold_metrics': fold_metrics,
            'training_time_mean': np.mean(training_times),
            'training_time_std': np.std(training_times),
            'inference_time_mean': np.mean(inference_times),
            'inference_time_std': np.std(inference_times),
            'inference_time_per_sample': np.mean(inference_times) / (len(y) / n_splits),
            'total_time': total_approach_time,
            'feature_dim': X.shape[1] if X is not None else 0
        }

        self.results[approach_name] = aggregated_metrics
        self.cv_predictions[approach_name] = {
            'y_true': np.array(all_y_true),
            'y_pred': np.array(all_y_pred),
            'y_pred_proba': np.array(all_y_pred_proba)
        }
        self.roc_data[approach_name] = roc_curves_data

        print(f"\n{'=' * 80}")
        print(f"FINAL RESULTS - {approach_name}")
        print(f"{'=' * 80}")
        print(f"Accuracy:    {aggregated_metrics['accuracy_mean']:.4f} ± {aggregated_metrics['accuracy_std']:.4f}")
        print(f"AUC:         {aggregated_metrics['auc_mean']:.4f} ± {aggregated_metrics['auc_std']:.4f}")

        return self

    def run_comprehensive_study(self, n_splits=5, include_fine_tuned=False,
                                use_mask_guided_deep_features=False):
        """
        Run comprehensive 

        Args:
            n_splits: Number of CV folds
            include_fine_tuned: Whether to include fine-tuned ResNet
            use_mask_guided_deep_features: Whether to use mask-guided deep features
        """
        print("\n" + "=" * 80)
        print("COMPREHENSIVE STUDY: Tumor Specific Features VS DEEP LEARNING")
        print(f"Using mask-guided deep features: {use_mask_guided_deep_features}")
        print("=" * 80)

        # Select which deep features to use
        if use_mask_guided_deep_features:
            if self.deep_features_with_mask is None:
                print("\n ERROR: Mask-guided deep features not extracted yet!")
                print("classifier.extract_deep_features(use_mask=True)")
                return self
            deep_features_to_use = self.deep_features_with_mask
            deep_label = "Deep Features (ResNet50 + Mask + XGBoost)"
        else:
            if self.deep_features is None:
                print("\n ERROR: Deep features not extracted")
                print(" classifier.extract_deep_features(use_mask=False)")
                return self
            deep_features_to_use = self.deep_features
            deep_label = "Deep Features (ResNet50 + XGBoost)"

        # 1. Specific Features Only
        self.evaluate_approach(
            "Specific Features Only (XGBoost)",
            self.radiomics_features,
            self.labels,
            classifier_type='xgboost',
            n_splits=n_splits
        )

        # 2. Deep Features Only
        self.evaluate_approach(
            deep_label,
            deep_features_to_use,
            self.labels,
            classifier_type='xgboost',
            n_splits=n_splits
        )

        # 3. Fusion
        fused_features = np.concatenate([
            self.radiomics_features,
            deep_features_to_use
        ], axis=1)

        fusion_label = "Fusion (Radiomics + Deep" + (" + Mask" if use_mask_guided_deep_features else "") + ", XGBoost)"
        self.evaluate_approach(
            fusion_label,
            fused_features,
            self.labels,
            classifier_type='xgboost',
            n_splits=n_splits
        )

        
        if include_fine_tuned:
            self.evaluate_approach(
                "Fine-Tuned ResNet50 (End-to-End)",
                None,
                self.labels,
                classifier_type='fine_tuned_resnet',
                n_splits=n_splits
            )

        print("\n" + "=" * 80)
        print("COMPREHENSIVE STUDY COMPLETE")
        print("=" * 80)

        return self

   

    def generate_summary_report(self):
        """Generate summary report"""
        print("\n" + "=" * 80)
        print("SUMMARY REPORT")
        print("=" * 80)

        approaches = list(self.results.keys())

        print("\n🏆 RANKING BY ACCURACY")
        sorted_by_acc = sorted(approaches, key=lambda x: self.results[x]['accuracy_mean'], reverse=True)
        for rank, approach in enumerate(sorted_by_acc, 1):
            acc = self.results[approach]['accuracy_mean']
            std = self.results[approach]['accuracy_std']
            auc_val = self.results[approach]['auc_mean']
            print(f"   {rank}. {approach:50s} Acc: {acc:.4f}±{std:.4f}  AUC: {auc_val:.4f}")

        return self


# CONFIGURATION CLASS - EDIT THIS TO CHANGE SETTINGS


class Config:
  

    # Paths
    CSV_PATH = 'Tumore_Specific_Features'
    DATA_ROOT = 'Brain_Dataset'

    # Experiment settings
    N_SPLITS = 5
    BATCH_SIZE = 32
    INCLUDE_FINE_TUNED = False

    # Deep feature extraction settings
    EXTRACT_FEATURES_WITHOUT_MASK = True    # Extract features without mask
    EXTRACT_FEATURES_WITH_MASK = True       # Extract features with mask

    USE_MASK_GUIDED_DEEP_FEATURES = False   # False = use without mask, True = use with mask


# MAIN EXECUTION

def main(config=None):
  

    if config is None:
        config = Config()

    print("=" * 80)
    print("MULTIMODAL BRAIN TUMOR CLASSIFICATION - COLAB VERSION")
    print("=" * 80)

    classifier = MultimodalBrainTumorClassifier(
        csv_path=config.CSV_PATH,
        data_root=config.DATA_ROOT,
        image_size=(224, 224)
    )

    result = classifier.load_and_prepare_data()

    if result is not None:
        # Extract deep features (with/without mask as specified)
        if config.EXTRACT_FEATURES_WITHOUT_MASK:
            print("\n Extracting features WITHOUT mask")
            classifier.extract_deep_features(
                batch_size=config.BATCH_SIZE,
                use_mask=False
            )

        if config.EXTRACT_FEATURES_WITH_MASK:
            print("\n Extracting features WITH mask")
            classifier.extract_deep_features(
                batch_size=config.BATCH_SIZE,
                use_mask=True
            )

    
        classifier.run_comprehensive_study(
            n_splits=config.N_SPLITS,
            include_fine_tuned=config.INCLUDE_FINE_TUNED,
            use_mask_guided_deep_features=config.USE_MASK_GUIDED_DEEP_FEATURES
        )

        # Generate report
        classifier.generate_summary_report()

        print("\nCOMPLETE!")
    else:
        print("\nFAILED.")


if __name__ == "__main__":
   
    main()